In [1]:
# Core
import numpy as np
import matplotlib.pyplot as plt

# Classical ML
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Metrics
from sklearn.metrics import accuracy_score, confusion_matrix

# Quantum
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.circuit.library import ZFeatureMap, TwoLocal
from qiskit_machine_learning.algorithms import VQC
from qiskit_machine_learning.neural_networks import SamplerQNN
from qiskit.algorithms.optimizers import COBYLA
from qiskit.primitives import Estimator
from qiskit_machine_learning.connectors import TorchConnector



/var/folders/g_/n2c03sss00zbmj6npbghnt_40000gn/T/ipykernel_58191/1631507250.py:21: DeprecationWarning: ``qiskit.algorithms`` has been migrated to an independent package: https://github.com/qiskit-community/qiskit-algorithms. The ``qiskit.algorithms`` import path is deprecated as of qiskit-terra 0.25.0 and will be removed no earlier than 3 months after the release date. Please run ``pip install qiskit_algorithms`` and use ``import qiskit_algorithms`` instead.
  from qiskit.algorithms.optimizers import COBYLA


In [15]:
IMAGE_SIZE = 200
BATCH_SIZE = 16

# Training transforms (with augmentation)
train_transform = transforms.Compose([
    transforms.Resize((200, 200)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.RandomResizedCrop(200, scale=(0.95, 1.0)),
    transforms.ToTensor(),
])

# Test transforms (NO augmentation)
test_transform = transforms.Compose([
    transforms.Resize((200, 200)),
    transforms.ToTensor(),
])


train_data = datasets.ImageFolder(
    "data/train", transform=train_transform
)
test_data = datasets.ImageFolder(
    "data/test", transform=test_transform
)

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False)

print("Classes:", train_data.classes)


Classes: ['covid', 'normal']


In [16]:
class ClassicalCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # (if you add more blocks later, no problem)
        )

        # Dynamically infer feature size
        with torch.no_grad():
            dummy = torch.zeros(1, 3, 200, 200)
            out = self.features(dummy)
            feature_dim = out.view(1, -1).size(1)

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(feature_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 1)   # logits
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


In [17]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)
model = ClassicalCNN().to(device)

class_counts = np.bincount(train_data.targets)
neg, pos = class_counts  # class 0, class 1
pos_weight = torch.tensor(neg / pos, dtype=torch.float32).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.StepLR(
    optimizer, step_size=15, gamma=0.5
)

for epoch in range(40):
    model.train()
    total_loss = 0

    epoch_loss = 0
    num_samples = 0

    for x, y in train_loader:
        x, y = x.to(device), y.float().to(device)

        # LABEL SMOOTHING
        y = y * 0.9 + 0.05

        optimizer.zero_grad()
        preds = model(x).squeeze()
        loss = criterion(preds, y)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * x.size(0)
        num_samples += x.size(0)

    print(f"Epoch {epoch+1}, Avg Loss: {epoch_loss / num_samples:.4f}")

    if epoch == 15:
        for param in model.features[0:4].parameters():
            param.requires_grad = False
        print("🔒 Frozen first conv block")

    scheduler.step()


Using device: mps
Epoch 1, Avg Loss: 0.7923
Epoch 2, Avg Loss: 0.5471
Epoch 3, Avg Loss: 0.5249
Epoch 4, Avg Loss: 0.5031
Epoch 5, Avg Loss: 0.4929
Epoch 6, Avg Loss: 0.4760
Epoch 7, Avg Loss: 0.4722
Epoch 8, Avg Loss: 0.4458
Epoch 9, Avg Loss: 0.4416
Epoch 10, Avg Loss: 0.4427
Epoch 11, Avg Loss: 0.4306
Epoch 12, Avg Loss: 0.4173
Epoch 13, Avg Loss: 0.4197
Epoch 14, Avg Loss: 0.4123
Epoch 15, Avg Loss: 0.4028
Epoch 16, Avg Loss: 0.3844
🔒 Frozen first conv block
Epoch 17, Avg Loss: 0.3782
Epoch 18, Avg Loss: 0.3703
Epoch 19, Avg Loss: 0.3692
Epoch 20, Avg Loss: 0.3634
Epoch 21, Avg Loss: 0.3603
Epoch 22, Avg Loss: 0.3546
Epoch 23, Avg Loss: 0.3534
Epoch 24, Avg Loss: 0.3504
Epoch 25, Avg Loss: 0.3570
Epoch 26, Avg Loss: 0.3467
Epoch 27, Avg Loss: 0.3498
Epoch 28, Avg Loss: 0.3468
Epoch 29, Avg Loss: 0.3391
Epoch 30, Avg Loss: 0.3435
Epoch 31, Avg Loss: 0.3268
Epoch 32, Avg Loss: 0.3206
Epoch 33, Avg Loss: 0.3172
Epoch 34, Avg Loss: 0.3228
Epoch 35, Avg Loss: 0.3160
Epoch 36, Avg Loss: 

In [18]:
# THRESHOLD SCAN (ANALYSIS)

from numpy import arange

model.eval()
probs_all, y_true = [], []

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        logits = model(x).squeeze()
        probs = torch.sigmoid(logits).cpu().numpy()
        probs_all.extend(probs.tolist())
        y_true.extend(y.numpy().tolist())

best_acc = 0
best_t = 0

for t in arange(0.35, 0.65, 0.01):
    preds = (np.array(probs_all) > t).astype(int)
    acc = accuracy_score(y_true, preds)
    if acc > best_acc:
        best_acc = acc
        best_t = t

print("Best threshold:", best_t)
print("Best CNN Accuracy:", best_acc)


Best threshold: 0.6100000000000002
Best CNN Accuracy: 0.73


In [19]:
FINAL_THRESHOLD = 0.65

model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        logits = model(x).squeeze()
        preds = (torch.sigmoid(logits) > FINAL_THRESHOLD).int().cpu().numpy()

        y_pred.extend(preds.tolist())
        y_true.extend(y.numpy().tolist())

cnn_acc = accuracy_score(y_true, y_pred)
print("Classical CNN Accuracy:", cnn_acc)
print(confusion_matrix(y_true, y_pred))


Classical CNN Accuracy: 0.7283333333333334
[[239  61]
 [102 198]]


**QCNN:**

In [29]:
class FeatureExtractor(nn.Module):
    def __init__(self, cnn_features):
        super().__init__()
        self.features = cnn_features

        with torch.no_grad():
            dummy = torch.zeros(1, 3, 200, 200).to(next(self.features.parameters()).device)
            out = self.features(dummy)
            feature_dim = out.view(1, -1).size(1)

        self.fc = nn.Linear(feature_dim, 4)  # quantum features

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)


In [30]:
# class HybridQCNN(nn.Module):
#     def __init__(self, qnn):
#         super().__init__()
#         self.qnn = qnn
#         self.fc = nn.Linear(16, 1)  # 🔑 2 observables → 1 logit

#     def forward(self, x):
#         x = self.qnn(x)
#         return self.fc(x)
class HybridQCNN(nn.Module):
    def __init__(self, qnn):
        super().__init__()
        self.qnn = qnn
        self.fc1 = nn.Linear(4, 16)
        self.act = nn.ReLU()
        self.fc2 = nn.Linear(16, 1)

    def forward(self, x):
        x = self.qnn(x)
        x = self.act(self.fc1(x))
        return self.fc2(x)



In [31]:
feature_model = FeatureExtractor(model).to(device)
feature_model.eval()

X_train, y_train = [], []

with torch.no_grad():
    for x, y in train_loader:
        x = x.to(device)
        features = feature_model(x).cpu().numpy()
        X_train.extend(features)
        y_train.extend(y.numpy()) 


In [32]:
# TRAIN FEATURE NORMALIZATION (PER-FEATURE)
X_train = np.array(X_train)
y_train = np.array(y_train)

# compute per-feature min/max on TRAIN ONLY
mins = X_train.min(axis=0)
maxs = X_train.max(axis=0)

# normalize each feature independently
X_train = (X_train - mins) / (maxs - mins + 1e-8)

# map to quantum angle range
X_train = X_train * np.pi   # [0, pi]

# BALANCE DATA FOR QCNN
from sklearn.utils import resample

# separate classes
pos = X_train[y_train == 1]
neg = X_train[y_train == 0]

# take equal samples
n = min(len(pos), len(neg))

X_bal = np.vstack([pos[:n], neg[:n]])
y_bal = np.hstack([np.ones(n), np.zeros(n)])

# shuffle
idx = np.random.permutation(len(X_bal))
X_bal = X_bal[idx]
y_bal = y_bal[idx]

# limit QCNN samples
X_train_q = X_bal[:200]
y_train_q = y_bal[:200]

print("QCNN class balance:", np.mean(y_train_q))  # should be ~0.5


QCNN class balance: 0.505


In [33]:
num_qubits = 4

feature_map = ZFeatureMap(num_qubits)
ansatz = TwoLocal(
    num_qubits,
    rotation_blocks=["ry", "rz"],
    entanglement_blocks="cx",
    reps=2,     # increase depth
    entanglement="full" # full or linear
)

qc = QuantumCircuit(num_qubits)
qc.compose(feature_map, inplace=True)
qc.compose(ansatz, inplace=True)
# qc.draw("mpl")

In [34]:
# backend = AerSimulator()
# optimizer = COBYLA(maxiter=100)
# estimator = Estimator()
from qiskit.primitives import Estimator
from qiskit.quantum_info import SparsePauliOp
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_aer.primitives import Estimator as AerEstimator

# estimator = AerEstimator(run_options={"shots": 1024})
from qiskit.primitives import Estimator
estimator = Estimator()   # analytic, NO shots

observables = [
    SparsePauliOp.from_list([("ZIII", 1.0)]),
    SparsePauliOp.from_list([("IZII", 1.0)]),
    SparsePauliOp.from_list([("IIZI", 1.0)]),
    SparsePauliOp.from_list([("IIIZ", 1.0)]),
]

qnn = EstimatorQNN(
    circuit=qc,
    observables=observables,
    input_params=feature_map.parameters,
    weight_params=ansatz.parameters,
    estimator=estimator,
)

base_qnn = TorchConnector(qnn).to(device)
model_qcnn = HybridQCNN(base_qnn).to(device)

# vqc.fit(X_train, y_train)


In [ ]:
optimizer_q = optim.Adam(model_qcnn.parameters(), lr=0.01) # If unstable: lr = 0.005
criterion_q = nn.BCEWithLogitsLoss()
X_q = torch.tensor(X_train_q, dtype=torch.float32).to(device)
y_q = torch.tensor(y_train_q, dtype=torch.float32).view(-1, 1).to(device)

batch_size = 16

for epoch in range(50):
    perm = torch.randperm(len(X_q))
    total_loss = 0

    for i in range(0, len(X_q), batch_size):
        idx = perm[i:i+batch_size]
        xb = X_q[idx]
        yb = y_q[idx]

        optimizer_q.zero_grad()
        outputs = model_qcnn(xb)
        loss = criterion_q(outputs, yb)
        loss.backward()
        optimizer_q.step()

        total_loss += loss.item()

    print(f"Epoch {epoch}, Loss: {total_loss:.4f}")

Epoch 0, Loss: 8.9976
Epoch 1, Loss: 8.7711
Epoch 2, Loss: 8.5460
Epoch 3, Loss: 8.0285
Epoch 4, Loss: 7.5803
Epoch 5, Loss: 7.0516
Epoch 6, Loss: 6.3192
Epoch 7, Loss: 5.9583
Epoch 8, Loss: 5.6931
Epoch 9, Loss: 5.2773
Epoch 10, Loss: 4.9876
Epoch 11, Loss: 4.3892
Epoch 12, Loss: 4.0805
Epoch 13, Loss: 3.6492
Epoch 14, Loss: 3.3751
Epoch 15, Loss: 3.5230
Epoch 16, Loss: 3.3378
Epoch 17, Loss: 3.1005
Epoch 18, Loss: 3.1246
Epoch 19, Loss: 2.7088


In [3]:
X_test, y_test = [], []

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        feats = feature_model(x).cpu().numpy()
        X_test.extend(feats)
        y_test.extend(y.numpy())

X_test = np.array(X_test)
y_test = np.array(y_test)

# NORMALIZE TEST USING TRAIN STATS
X_test = (X_test - mins) / (maxs - mins + 1e-8)
X_test = X_test * np.pi


NameError: name 'test_loader' is not defined

In [30]:
# Convert to torch
X_t = torch.tensor(X_test, dtype=torch.float32).to(device)

model_qcnn.eval()

with torch.no_grad():
    outputs = model_qcnn(X_t)
    probs = torch.sigmoid(outputs).cpu().numpy()
    preds = (probs > 0.5).astype(int)

acc = accuracy_score(y_test, preds)
print("QCNN Accuracy:", acc)


QCNN Accuracy: 0.6816666666666666
